# Aula 04 — Streaming de Eventos de Varejo

Demonstração do pipeline de event streaming com **Redpanda/Kafka** e camadas **Bronze / Silver / Gold**.

```
publisher.py  →  Redpanda/Kafka  →  subscriber.py  →  data/bronze
                                                    →  data/silver
                                                    →  data/gold
```

| Camada | Arquivo | O que contém |
|--------|---------|-------------|
| Bronze | `data/bronze/events.jsonl` | Evento cru como chegou do broker |
| Silver | `data/silver/events_normalized.jsonl` | Schema normalizado, campos homogêneos |
| Gold | `data/gold/metrics_snapshot.json` | Agregados em tempo quase real |

---
## Ato 1 — Infraestrutura

Sobe o broker **Redpanda** e o **Console** web.

> Após executar, abra http://localhost:8080 — ainda não há tópico nem mensagem.

In [ ]:
!docker compose up -d

In [ ]:
!docker compose ps

---
## Ato 2 — Entendendo o código

### publisher.py — o produtor

Gera eventos simulados de varejo e publica no tópico `varejo.eventos`.

- **3 tipos de evento:** `order_created`, `payment_authorized`, `inventory_updated`
- Cada evento tem: `event_id`, `event_type`, `event_time` + `payload` específico
- Parâmetros configuráveis: `--messages` e `--interval`

### subscriber.py — o consumidor

Consome o tópico e para cada mensagem faz **3 coisas em sequência**:

1. Grava no **Bronze** (dado cru)
2. Normaliza e grava no **Silver** (schema comum)
3. Atualiza o **Gold** (métricas acumuladas)

> O broker é o "corredor" — publisher e subscriber não se comunicam diretamente.

In [ ]:
# Estrutura dos eventos gerados pelo publisher
import json, uuid
from datetime import datetime, timezone

exemplos = {
    "order_created": {
        "event_id": str(uuid.uuid4()),
        "event_type": "order_created",
        "event_time": datetime.now(timezone.utc).isoformat(),
        "payload": {
            "order_id": str(uuid.uuid4()),
            "customer_id": "CUST-4231",
            "store_id": "JP-CENTRO",
            "channel": "ecommerce",
            "items": [{"sku": "SKU-NOTEBOOK", "quantity": 1, "unit_price": 299.90}],
            "total_amount": 299.90
        }
    },
    "payment_authorized": {
        "event_id": str(uuid.uuid4()),
        "event_type": "payment_authorized",
        "event_time": datetime.now(timezone.utc).isoformat(),
        "payload": {"order_id": str(uuid.uuid4()), "payment_method": "pix", "amount": 299.90, "status": "approved"}
    },
    "inventory_updated": {
        "event_id": str(uuid.uuid4()),
        "event_type": "inventory_updated",
        "event_time": datetime.now(timezone.utc).isoformat(),
        "payload": {"sku": "SKU-NOTEBOOK", "store_id": "JP-CENTRO", "delta": -1, "quantity_after": 7}
    }
}

for tipo, evento in exemplos.items():
    print(f"--- {tipo} ---")
    print(json.dumps(evento, indent=2, ensure_ascii=False))
    print()

---
## Ato 3 — Demo ao vivo

Abra **dois terminais** separados na raiz do projeto.

**Terminal 1** — subscriber (fica aguardando mensagens):
```bash
source .venv/bin/activate
python subscriber.py
```

**Terminal 2** — publisher (publica com intervalo lento para acompanhar):
```bash
source .venv/bin/activate
python publisher.py --messages 40 --interval 0.8
```

> Observe no Terminal 1: cada linha mostra `offset`, `orders`, `approved`, `revenue` atualizando em tempo real.  
> Observe no Console web (http://localhost:8080): o tópico `varejo.eventos` acumulando mensagens.

---
## Ato 4 — Inspecionar as camadas com DuckDB

DuckDB lê os arquivos JSONL/JSON diretamente, sem banco de dados.

> Execute as células abaixo **enquanto o subscriber está rodando** para ver os dados crescendo.

In [1]:
import duckdb
from pathlib import Path

BRONZE = "data/bronze/events.jsonl"
SILVER = "data/silver/events_normalized.jsonl"
GOLD   = "data/gold/metrics_snapshot.json"

# Confirmar que os arquivos existem
for path in [BRONZE, SILVER, GOLD]:
    p = Path(path)
    status = f"{p.stat().st_size} bytes" if p.exists() else "nao encontrado"
    print(f"{path}: {status}")

data/bronze/events.jsonl: 76805 bytes
data/silver/events_normalized.jsonl: 81137 bytes
data/gold/metrics_snapshot.json: 507 bytes


### Camada Bronze — dado cru

Os eventos chegam **exatamente como foram publicados**. Payloads heterogêneos: campos diferentes por tipo de evento.

> Propósito: auditoria, rastreabilidade, reprocessamento futuro.

In [74]:
# Contagem por tipo de evento
duckdb.sql(f"""
    SELECT event_type, COUNT(*) AS total
    FROM '{BRONZE}'
    GROUP BY event_type
    ORDER BY total DESC
""").df()

,event_type,total
0,order_created,63
1,payment_authorized,22
2,inventory_updated,15


In [76]:
# Últimos 5 eventos (observe o campo payload heterogêneo)
duckdb.sql(f"""
    SELECT event_type, event_time,
           payload.order_id,
           payload.channel,
           payload.total_amount,
           payload.status,
           payload.sku,
           payload.quantity_after
    FROM '{BRONZE}'
    ORDER BY event_time DESC
    LIMIT 5
""").df()

,event_type,event_time,order_id,channel,total_amount,status,sku,quantity_after
0,payment_authorized,2026-04-07T00:01:45.618976+00:00,2fa40c64-e5c3-4eea-81ab-b0626dca2545,NaN,NaN,approved,NaN,<NA>
1,payment_authorized,2026-04-07T00:01:45.113204+00:00,2fa40c64-e5c3-4eea-81ab-b0626dca2545,NaN,NaN,denied,NaN,<NA>
2,inventory_updated,2026-04-07T00:01:44.602916+00:00,<NA>,NaN,NaN,NaN,SKU-LIVRO,78
3,order_created,2026-04-07T00:01:44.098269+00:00,e6be185f-ef5c-4d7a-a4f4-84b5a448080a,ecommerce,389.98,NaN,NaN,<NA>
4,payment_authorized,2026-04-07T00:01:43.591105+00:00,cd633088-f646-4044-a875-276088b21102,NaN,NaN,approved,NaN,<NA>


### Camada Silver — schema normalizado

Os 3 tipos de evento agora têm **campos comuns**: `order_id`, `store_id`, `amount`, `status`.  
Campos ausentes para um tipo são `NULL` explícito — não ficam escondidos dentro do payload.

> Propósito: consultas consistentes sem precisar conhecer o schema de cada tipo de evento.

In [77]:
# Últimos 5 eventos normalizados — compare com o Bronze acima
duckdb.sql(f"""
    SELECT event_type, order_id, store_id, channel, amount, status, processed_at
    FROM '{SILVER}'
    ORDER BY event_time DESC
    LIMIT 5
""").df()

,event_type,order_id,store_id,channel,amount,status,processed_at
0,payment_authorized,2fa40c64-e5c3-4eea-81ab-b0626dca2545,NaN,NaN,378.45,approved,2026-04-07T00:01:45.625974+00:00
1,payment_authorized,2fa40c64-e5c3-4eea-81ab-b0626dca2545,NaN,NaN,165.30,denied,2026-04-07T00:01:45.116022+00:00
2,inventory_updated,<NA>,CG-01,NaN,NaN,inventory_changed,2026-04-07T00:01:44.609161+00:00
3,order_created,e6be185f-ef5c-4d7a-a4f4-84b5a448080a,JP-CENTRO,ecommerce,389.98,created,2026-04-07T00:01:44.100271+00:00
4,payment_authorized,cd633088-f646-4044-a875-276088b21102,NaN,NaN,261.01,approved,2026-04-07T00:01:43.593567+00:00


In [78]:
# Ticket médio por canal (consulta que seria impossível direto no Bronze sem explodir o payload)
duckdb.sql(f"""
    SELECT channel,
           COUNT(*)              AS pedidos,
           ROUND(AVG(amount), 2) AS ticket_medio
    FROM '{SILVER}'
    WHERE event_type = 'order_created'
      AND channel IS NOT NULL
    GROUP BY channel
    ORDER BY ticket_medio DESC
""").df()

,channel,pedidos,ticket_medio
0,loja_fisica,18,1161.67
1,ecommerce,25,893.64
2,app,20,741.28


In [6]:
# Pagamentos aprovados vs negados
duckdb.sql(f"""
    SELECT status,
           COUNT(*)              AS total,
           ROUND(SUM(amount), 2) AS valor_total
    FROM '{SILVER}'
    WHERE event_type = 'payment_authorized'
    GROUP BY status
""").df()

,status,total,valor_total
0,approved,51,17872.26
1,denied,5,1367.84


In [7]:
# Alertas de estoque crítico (quantity_after <= 10)
duckdb.sql(f"""
    SELECT store_id, sku, delta, quantity_after
    FROM '{SILVER}'
    WHERE event_type = 'inventory_updated'
      AND quantity_after <= 10
    ORDER BY quantity_after
""").df()

,store_id,sku,delta,quantity_after
0,CG-01,SKU-ARROZ,5.0,0
1,CG-01,SKU-NOTEBOOK,-6.0,2
2,CG-01,SKU-ARROZ,-4.0,2
3,JP-CENTRO,SKU-LIVRO,-4.0,6
4,CG-01,SKU-ARROZ,4.0,6
5,JP-CENTRO,SKU-ARROZ,12.0,7
6,CG-01,SKU-FONE,0.0,8


### Camada Gold — métricas agregadas

Um único arquivo JSON, **sobrescrito a cada evento**, com os indicadores acumulados.

> Propósito: alimentar dashboards e alertas em tempo quase real — sem consultar o histórico completo.

In [8]:
# KPIs principais
duckdb.sql(f"""
    SELECT generated_at,
           total_orders,
           payments_approved,
           payments_denied,
           approval_rate_percent,
           approved_revenue
    FROM '{GOLD}'
""").df()

,generated_at,total_orders,payments_approved,payments_denied,approval_rate_percent,approved_revenue
0,2026-04-05T17:24:16.343908+00:00,111,51,5,91.07,17872.26


In [9]:
# Receita aprovada por loja
duckdb.sql(f"""
    SELECT
        unnest(map_keys(approved_revenue_per_store))              AS store_id,
        ROUND(unnest(map_values(approved_revenue_per_store)), 2)  AS receita_aprovada
    FROM read_json('{GOLD}', columns={{'approved_revenue_per_store': 'MAP(VARCHAR, DOUBLE)'}})
    ORDER BY receita_aprovada DESC
""").df()

,store_id,receita_aprovada
0,JP-CENTRO,8285.40
1,JP-SUL,6816.65
2,CG-01,2088.38
3,UNKNOWN,681.83


In [10]:
# Pedidos por canal
duckdb.sql(f"""
    SELECT
        unnest(map_keys(orders_per_channel))   AS canal,
        unnest(map_values(orders_per_channel)) AS pedidos
    FROM read_json('{GOLD}', columns={{'orders_per_channel': 'MAP(VARCHAR, BIGINT)'}})
    ORDER BY pedidos DESC
""").df()

,canal,pedidos
0,app,42
1,ecommerce,36
2,loja_fisica,33


---
## Ato 5 — Rastreando um evento nas 3 camadas

O mesmo `event_id` percorre Bronze → Silver → Gold.  
Abaixo rastreamos o **primeiro evento** do arquivo para mostrar o que muda em cada camada.

In [11]:
# Pega o primeiro event_id do Bronze
event_id = duckdb.sql(f"SELECT event_id FROM '{BRONZE}' LIMIT 1").fetchone()[0]
print(f"Rastreando event_id: {event_id}")

Rastreando event_id: 806f40dd-433c-4f88-9e17-7e532f114ba9


In [12]:
# BRONZE — evento cru com payload aninhado
print("BRONZE — dado cru:")
duckdb.sql(f"""
    SELECT event_id, event_type, event_time,
           payload.sku, payload.store_id, payload.delta, payload.quantity_after,
           payload.order_id, payload.channel, payload.total_amount
    FROM '{BRONZE}'
    WHERE event_id = '{event_id}'
""").df()

BRONZE — dado cru:


,event_id,event_type,event_time,sku,store_id,delta,quantity_after,order_id,channel,total_amount
0,806f40dd-433c-4f88-9e17-7e532f114ba9,order_created,2026-04-05T16:30:46.316884+00:00,None,JP-SUL,NaN,<NA>,50d6febc-3a07-413e-8a80-b5752b1ff3ac,ecommerce,93.77


In [13]:
# SILVER — mesmo evento, schema normalizado e processed_at adicionado
print("SILVER — normalizado:")
duckdb.sql(f"""
    SELECT event_id, event_type, order_id, store_id, amount, status, sku, processed_at
    FROM '{SILVER}'
    WHERE event_id = '{event_id}'
""").df()

SILVER — normalizado:


,event_id,event_type,order_id,store_id,amount,status,sku,processed_at
0,806f40dd-433c-4f88-9e17-7e532f114ba9,order_created,50d6febc-3a07-413e-8a80-b5752b1ff3ac,JP-SUL,93.77,created,None,2026-04-05T17:23:12.848512+00:00


In [35]:
# GOLD — snapshot acumulado após todos os eventos incluindo este
print("GOLD — estado agregado após este e todos os eventos anteriores:")
duckdb.sql(f"""
    SELECT total_orders, payments_approved, payments_denied,
           approval_rate_percent, approved_revenue
    FROM '{GOLD}'
""").df()

GOLD — estado agregado após este e todos os eventos anteriores:


,total_orders,payments_approved,payments_denied,approval_rate_percent,approved_revenue
0,170,74,8,90.24,25290.24


---
## Ato 6 — Volume maior e atualização contínua

Limpa os dados e publica com maior volume para mostrar a atualização contínua do Gold.

> Mantenha o **subscriber rodando** no Terminal 1 e execute o publisher abaixo.

In [15]:
# Limpar dados anteriores
import shutil, os
shutil.rmtree("data", ignore_errors=True)
for d in ["data/bronze", "data/silver", "data/gold"]:
    os.makedirs(d, exist_ok=True)
print("Dados limpos.")

Dados limpos.


In [84]:
# Publicar 100 eventos com intervalo rápido
!python publisher.py --messages 500 --interval 0.2

Conectando em localhost:19092 e publicando em 'varejo.eventos'
[001] order_created -> {"order_id": "97367b3e-8505-4b49-83e4-193fd39c86dc", "customer_id": "CUST-8233", "store_id": "JP-CENTRO", "channel": "loja_fisica", "items": [{"sku": "SKU-NOTEBOOK", "quantity": 2, "unit_price": 127.93}, {"sku": "SKU-ARROZ", "quantity": 1, "unit_price": 281.89}], "total_amount": 537.75}
[002] order_created -> {"order_id": "7e7dabec-289a-406e-84d1-830c78842f01", "customer_id": "CUST-9115", "store_id": "CG-01", "channel": "ecommerce", "items": [{"sku": "SKU-ARROZ", "quantity": 1, "unit_price": 192.59}, {"sku": "SKU-CAFE", "quantity": 2, "unit_price": 252.05}, {"sku": "SKU-ARROZ", "quantity": 1, "unit_price": 230.71}], "total_amount": 927.4}
[003] order_created -> {"order_id": "ee6e5f0a-5b2e-4b90-ad76-982bdde1f0bc", "customer_id": "CUST-9457", "store_id": "CG-01", "channel": "loja_fisica", "items": [{"sku": "SKU-FONE", "quantity": 3, "unit_price": 158.22}, {"sku": "SKU-LIVRO", "quantity": 1, "unit_price"

In [131]:
# Re-executar esta célula várias vezes enquanto o publisher roda para ver o Gold atualizando
duckdb.sql(f"""
    SELECT generated_at, total_orders, payments_approved,
            approval_rate_percent, approved_revenue
    FROM '{GOLD}'
""").df()

,generated_at,total_orders,payments_approved,approval_rate_percent,approved_revenue
0,2026-04-07T00:19:15.446701+00:00,117,51,75.0,17740.68


---
## Encerrar

Pare o subscriber com `Ctrl+C` no Terminal 1 e derrube os containers:

In [ ]:
!docker compose down

---
## Resumo

| Conceito | O que foi demonstrado |
|----------|-----------------------|
| **Desacoplamento** | Publisher e subscriber se comunicam apenas via broker |
| **Durabilidade** | O offset garante que nenhuma mensagem se perde |
| **Bronze** | Evento cru — payload heterogêneo por tipo, ideal para reprocessamento |
| **Silver** | Schema normalizado — consultas SQL diretas sem explodir payloads |
| **Gold** | Agregados atualizados a cada evento — base para dashboards |
| **DuckDB** | Consulta JSONL/JSON como tabela sem banco de dados externo |